# Transform data from bronze layer

## 1. Import related module

In [0]:
import sys
sys.path.append("/Workspace/Users/sophonsaesim@gmail.com/ReadCSVdata_2026/readcsvdata/configs")

In [0]:
import pyspark.sql.functions as F           # import pyspark function
import silver_config as sc                  # import silver configuration filr
from pyspark.sql.types import StringType    # import pyspark string type 
import importlib                            # import reload library

importlib.reload(sc)

## 2. Transform data process

In [0]:
# drop duplicates key column
def drop_dup_key(silver_df):
    return silver_df.dropDuplicates(["cst_id"])

### Function : Removed leading and tailing unwanted space

In [0]:
# Function : Remove unwanted space
def trim_string_col(silver_df, col_type, null_value="Unknown"):
    # remove leading and tailing unwanted space
        return (
            silver_df.withColumn(col_type.name,
                                 F.when(F.col(col_type.name).isNotNull(), F.trim(F.col(col_type.name)))
                                 .otherwise(null_value)
                                 )
        )

### Function : Normolized spacific column

In [0]:

# Marital status normallization
def marital_normalize(silver_df, marital_col, marital_map, null_value="unknown", invalid="Invalid"):
    # Flatten marital status dictionaty to list
    marital_expr =  F.create_map(
        [F.lit(marital_flat) for marital_kv in marital_map.items() for marital_flat in marital_kv]
    )
    # Return transformed marital status
    return silver_df.withColumn(marital_col, 
                                F.when(F.col(marital_col).isNull(), F.lit(null_value))
                                .otherwise(F.coalesce(marital_expr[F.upper(F.col(marital_col))],
                                                      F.lit(invalid))
                                        )
    )

In [0]:
def gender_normalize(silver_df, gender_col, gender_map, null_value="Unknown", invalid="Invalid"):
    # Flatten gender dictionary to list
    gender_expr = F.create_map(
        [F.lit(gender_flat) for gender_kv in gender_map.items() for gender_flat in gender_kv]
    )
    # Return transformed gender
    return (
        silver_df.withColumn(gender_col, 
                             F.when(F.col(gender_col).isNull(), F.lit(null_value))
                             .otherwise(F.coalesce(gender_expr[F.upper(F.col(gender_col))],
                                                            F.lit(invalid)
                                                  )
                                        )
        )
    )

### Function : Rename column

In [0]:
# Function : Rename column
def rename_col(silver_df, current_name, new_name):
    return (
        silver_df.withColumnRenamed(current_name, new_name)          
    )

### Function : Silver layer transformation data main program

In [0]:
# silver layer data transformation process main program
def transform_silver(silver_df):
    silver_res = "OK"

    # drop duplicates key column
    silver_df = drop_dup_key(silver_df)

    # remove leading and tailing unwanted space
    for col_type in silver_df.schema.fields:            
        if isinstance(col_type.dataType, StringType):
            silver_df = trim_string_col(silver_df, col_type)

    # normallize marital status
    silver_df = marital_normalize(silver_df, "cst_marital_status", sc.MARITAL_MAPPING)

    # normallize gender 
    silver_df = gender_normalize(silver_df, "cst_gndr", sc.GENDER_MAPPING)

    # rename column
    for current_name, new_name in sc.COLUMN_MAPPING.items():
        silver_df = rename_col(silver_df, current_name, new_name)

    if silver_res == "OK":
        pass
    return silver_df
    

## 3. Load transformed data to silver layer

In [0]:
#Function : Write completed transformed data to silver table
def write_silver(silver_df): 
    # write silver layer table
    return(
        silver_df.write.mode("append")
                        .format("delta")
                        .option("mergeSchema", "true")
                        .saveAsTable("readcsvdata.silver.silver_customer_info")
    )

## 4. Quality check transformed of silver layer

In [0]:
def qualitycheck_silver(silver_df):
    # duplicated key column
    silver_df.groupBy("customer_key").count().filter("count > 1").show()

    # check no unwanted space of string column
    silver_df.select(silver_df["customer_firstname"]).filter("trim(customer_key) != customer_key").show()
    silver_df.select(silver_df["customer_lastname"]).filter("trim(customer_key) != customer_key").show()

    # distinct data normallization
    silver_df.select(silver_df["customer_marital_status"]).distinct().show()
    silver_df.select(silver_df["customer_gender"]).distinct().show()

## Function : Silver transformation data process main program

In [0]:
def exec_silver():
    # read data from bronze table
    silver_df = spark.table("readcsvdata.bronze.bronze_crm_cust_info")
    
    # transform bronze dataframe
    silver_df = transform_silver(silver_df)

    # write silver dataframe to delta table
    silver_df = silver_df.withColumn("current_time", 
                                     F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
    write_silver(silver_df)

    # quality check table
    qualitycheck_silver(silver_df)
    return silver_df

## Execute silver layer transformation

In [0]:
exec_silver = exec_silver()